# PVision AI: Photovoltaic Fault Classification with EfficientNetV2-S
### A Hybrid CNN-ANN Framework for Photovoltaic Fault Detection and Power Generation Prediction
**Kaggle GPU Execution Notebook**

This notebook implements the complete transfer learning, fine-tuning, evaluation, and diagnostic visualization pipeline for the Vision Diagnostics branch of **PVision AI** using **EfficientNetV2-S**.

---
### Key Highlights:
- **Primary Model:** EfficientNetV2-S with ImageNet-1k pretrained weights
- **Dual Modality Support:** GASF (2D time-series matrices) and I-V characteristic curves
- **Aspect-Ratio-Preserving Padding:** Prevents geometric stretching of I-V curves
- **Stratified Partitioning:** Zero-leakage 70% Train / 15% Val / 15% Test splits
- **Two-Stage Training:** Stage 1 (Frozen Backbone) + Stage 2 (Unfreeze Top 50 Layers)
- **Output Artifacts:** Checkpoint (.keras), Confusion Matrix, Loss/Accuracy curves, and per-class metrics saved directly to `/kaggle/working/`

## 1. Import Libraries

In [ ]:
import os
import sys
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers, losses, metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

print(f"TensorFlow Version: {tf.__version__}")
print(f"NumPy Version:      {np.__version__}")
print(f"Pandas Version:     {pd.__version__}")

## 2. Kaggle GPU Verification
Audits the active Kaggle accelerator and enables dynamic GPU memory growth.

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
print("=" * 65)
print("             HARDWARE ACCELERATION AUDIT")
print("=" * 65)
print(f"TensorFlow Version: {tf.__version__}")

if gpus:
    print(f"GPU Available:      YES ({len(gpus)} device(s) detected)")
    for idx, gpu in enumerate(gpus):
        print(f"  [{idx}] Device Name: {gpu.name}")
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
            print(f"  [{idx}] Memory Growth: ENABLED")
        except Exception as e:
            print(f"  [{idx}] Memory Growth Note: {e}")
    active_gpu = tf.test.gpu_device_name()
    print(f"Active Device:      {active_gpu}")
else:
    print("GPU Available:      NO (WARNING: Training on CPU will be slow)")
    print("RECOMMENDATION:     In the Kaggle Notebook sidebar (Settings -> Accelerator),")
    print("                    select 'GPU P100' or 'GPU T4 x2'.")
print("=" * 65)

## 3. Configuration & Hyperparameters
Central configuration specifying input resolution, batch size, learning rates, epochs, and class labels.

In [ ]:
# ==============================================================================
# PVISION AI - HYPERPARAMETER & ENVIRONMENT CONFIGURATION
# ==============================================================================

# Deterministic Random Seed for Reproducibility
RANDOM_SEED = 42

# Image Dimensions & Batching
IMAGE_SIZE = (224, 224)          # EfficientNetV2-S standard input resolution
INPUT_SHAPE = (224, 224, 3)
BATCH_SIZE = 32

# Two-Stage Transfer Learning Configuration
EPOCHS_STAGE1 = 10               # Stage 1: Feature Extraction (Backbone frozen)
LR_STAGE1 = 1e-3                 # Learning rate for custom head

EPOCHS_STAGE2 = 20               # Stage 2: Fine-Tuning (Upper layers unfrozen)
LR_STAGE2 = 1e-4                 # Reduced learning rate for fine-tuning
UNFREEZE_LAYERS = 50             # Top N backbone layers to unfreeze

# Partitioning Ratios (70% Train / 15% Val / 15% Test)
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Class Weighting Toggle (helpful for I-V short_circuit imbalance)
USE_CLASS_WEIGHTS = True

# Exact 7 Photovoltaic Fault Categories
CLASS_NAMES = [
    "crack",
    "global_aging",
    "hotspot",
    "normal",
    "partial_aging",
    "shading",
    "short_circuit"
]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for idx, name in enumerate(CLASS_NAMES)}

# Output Directories on Kaggle
WORKING_DIR = Path("/kaggle/working")
RESULTS_DIR = WORKING_DIR / "results" / "cnn"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Set deterministic random seeds
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("Configuration initialized successfully.")
print(f"Target Fault Classes ({NUM_CLASSES}): {CLASS_NAMES}")

## 4. Dataset Path Configuration
Configure the path to your Kaggle dataset. Modify `DATASET_PATH` below to match your attached input dataset.

In [ ]:
# ==============================================================================
# KAGGLE DATASET INPUT PATH
# Modify this path according to your attached Kaggle input directory.
# Example: /kaggle/input/pvision-ai/data/pv_fault/gasf_images
# ==============================================================================

DATASET_PATH = "/kaggle/input/pvision-ai/data/pv_fault/gasf_images"
MODALITY = "gasf"  # Options: 'gasf' or 'iv'

dataset_dir = Path(DATASET_PATH)

# Automatic fallback check for parent pv_fault directory
if not dataset_dir.exists():
    alt_candidate = dataset_dir / ("gasf_images" if MODALITY == "gasf" else "iv_images")
    if alt_candidate.exists():
        dataset_dir = alt_candidate
    else:
        raise FileNotFoundError(
            f"ERROR: Dataset path does not exist: {DATASET_PATH}\n\n"
            f"Action Required:\n"
            f"1. Click '+ Add Input' in the Kaggle right sidebar to attach your dataset.\n"
            f"2. Run '!ls /kaggle/input/' to inspect available folder names.\n"
            f"3. Update DATASET_PATH above to match your Kaggle dataset path."
        )

print(f"Active Dataset Directory: {dataset_dir}")
print(f"Configured Modality:      {MODALITY.upper()}")

## 5. Dataset Structure Inspection & File Discovery
Scans the dataset directory, validates class subfolders, and records sample counts.

In [ ]:
image_records = []

print("Scanning dataset directories...")
print("-" * 65)
for cls_name in CLASS_NAMES:
    cls_folder = dataset_dir / cls_name
    if not cls_folder.is_dir():
        print(f"Warning: Subfolder not found for class: {cls_name}")
        continue
    
    files = [f for f in cls_folder.iterdir() if f.is_file() and f.suffix.lower() in [".png", ".jpg", ".jpeg"]]
    sample_file = files[0].name if files else "N/A"
    print(f"Class: {cls_name:<16} | Samples: {len(files):>5,} | Sample File: {sample_file}")
    
    for f in files:
        image_records.append({
            "file_path": str(f),
            "filename": f.name,
            "class_name": cls_name,
            "label": CLASS_TO_IDX[cls_name]
        })

df_all = pd.DataFrame(image_records)

if df_all.empty:
    raise ValueError(f"No image files (.png/.jpg) found in {dataset_dir}. Verify dataset path.")

print("-" * 65)
print(f"Total Images Discovered: {len(df_all):,} across {df_all['class_name'].nunique()} classes.")

## 6. Class Distribution Visualization
Visualizes sample counts per class to verify balance.

In [ ]:
plt.figure(figsize=(10, 5))
class_counts = df_all["class_name"].value_counts().reindex(CLASS_NAMES)
palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2"]

bars = plt.bar(class_counts.index, class_counts.values, color=palette, edgecolor="black", alpha=0.85)
plt.title(f"PV Fault Class Distribution — {MODALITY.upper()} Modality\nTotal: {len(df_all):,} Images", fontsize=13, fontweight="bold")
plt.xlabel("PV Fault Classes", fontsize=11, fontweight="bold")
plt.ylabel("Number of Samples", fontsize=11, fontweight="bold")
plt.xticks(rotation=25, ha="right", fontsize=10)
plt.grid(axis="y", linestyle="--", alpha=0.6)

for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        yval + (max(class_counts) * 0.01),
        f"{int(yval):,}",
        ha="center", va="bottom", fontsize=9, fontweight="bold"
    )

plt.tight_layout()
plt.savefig(RESULTS_DIR / "class_distribution.png", dpi=200)
plt.savefig(WORKING_DIR / "class_distribution.png", dpi=200)
plt.show()

# Tabular Breakdown
df_distribution = pd.DataFrame({
    "Class Name": class_counts.index,
    "Sample Count": class_counts.values,
    "Proportion (%)": np.round((class_counts.values / len(df_all)) * 100, 2)
})
display(df_distribution)

## 7. Sample Image Visualizations
Displays representative samples from each fault category to inspect visual features and native resolutions.

In [ ]:
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(20, 3.5))

for idx, cls_name in enumerate(CLASS_NAMES):
    cls_samples = df_all[df_all["class_name"] == cls_name]["file_path"].values
    if len(cls_samples) > 0:
        sample_img = Image.open(cls_samples[0])
        axes[idx].imshow(sample_img)
        axes[idx].set_title(f"{cls_name}\n({sample_img.size[0]}x{sample_img.size[1]})", fontsize=10, fontweight="bold")
    axes[idx].axis("off")

plt.suptitle(f"PV Fault Sample Gallery — {MODALITY.upper()} Modality", fontsize=13, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

## 8. Train / Validation / Test Dataset Partitioning (70% / 15% / 15%)
Stratified splitting to preserve the 7-class distribution while preventing data leakage.

In [ ]:
# Stratified Splitting
train_df, temp_df = train_test_split(
    df_all,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df_all["label"],
    random_state=RANDOM_SEED
)

relative_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_ratio,
    stratify=temp_df["label"],
    random_state=RANDOM_SEED
)

print("=" * 60)
print("            DATASET PARTITIONING AUDIT")
print("=" * 60)
print(f"Training Samples:   {len(train_df):>6,} ({len(train_df)/len(df_all)*100:>5.1f}%)")
print(f"Validation Samples: {len(val_df):>6,} ({len(val_df)/len(df_all)*100:>5.1f}%)")
print(f"Testing Samples:    {len(test_df):>6,} ({len(test_df)/len(df_all)*100:>5.1f}%)")
print(f"Total:              {len(df_all):>6,} (100.0%)")
print("=" * 60)

# Verify zero data leakage
train_set = set(train_df["file_path"])
val_set = set(val_df["file_path"])
test_set = set(test_df["file_path"])
assert len(train_set.intersection(val_set)) == 0, "Leakage detected between Train and Val!"
assert len(train_set.intersection(test_set)) == 0, "Leakage detected between Train and Test!"
print("Zero data leakage verified across all three splits.")

# Compute balanced class weights
counts = np.bincount(train_df["label"].values, minlength=NUM_CLASSES)
total_train = len(train_df)
class_weights = {i: float(total_train / (NUM_CLASSES * counts[i])) for i in range(NUM_CLASSES)}

print("\nComputed Class Weights (Active if USE_CLASS_WEIGHTS=True):")
for c_idx, w in class_weights.items():
    print(f"  Class {c_idx} ({CLASS_NAMES[c_idx]:<14}): {w:.4f}")

## 9. Image Preprocessing & Aspect-Ratio-Preserving Pipeline
- **GASF Modality:** Bilinear resize directly from $256 	imes 256$ to $224 	imes 224$
- **I-V Modality:** Aspect-ratio-preserving resize and padding to $224 	imes 224$ (avoids geometric curve stretching)
- **Data Augmentation:** Horizontal/vertical flips, subtle brightness/contrast variations applied strictly to the training split.

In [ ]:
is_iv = "iv" in MODALITY.lower()
target_h, target_w = IMAGE_SIZE

def preprocess_image(file_path, label, is_training: bool):
    img_bytes = tf.io.read_file(file_path)
    img = tf.image.decode_png(img_bytes, channels=3)
    img = tf.cast(img, tf.float32)

    if is_iv:
        # Aspect-ratio-preserving resize and pad (Zero distortion!)
        img = tf.image.resize_with_pad(
            img,
            target_height=target_h,
            target_width=target_w,
            method=tf.image.ResizeMethod.BILINEAR
        )
    else:
        # Square GASF direct bilinear resize
        img = tf.image.resize(
            img,
            [target_h, target_w],
            method=tf.image.ResizeMethod.BILINEAR
        )

    # Augmentation suite for training only
    if is_training:
        img = tf.image.random_flip_left_right(img, seed=RANDOM_SEED)
        img = tf.image.random_flip_up_down(img, seed=RANDOM_SEED)
        img = tf.image.random_brightness(img, max_delta=0.08, seed=RANDOM_SEED)
        img = tf.image.random_contrast(img, lower=0.92, upper=1.08, seed=RANDOM_SEED)

    img = tf.clip_by_value(img, 0.0, 255.0)
    label_one_hot = tf.one_hot(label, depth=NUM_CLASSES)
    return img, label_one_hot

def build_dataset(df, is_training: bool):
    paths = tf.constant(df["file_path"].values, dtype=tf.string)
    labels = tf.constant(df["label"].values, dtype=tf.int32)
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if is_training:
        ds = ds.shuffle(buffer_size=min(len(df), 5000), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    
    ds = ds.map(lambda p, l: preprocess_image(p, l, is_training), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

train_ds = build_dataset(train_df, is_training=True)
val_ds = build_dataset(val_df, is_training=False)
test_ds = build_dataset(test_df, is_training=False)

print("Optimized tf.data input pipelines ready for GPU training.")

## 10. EfficientNetV2-S Model Creation
Loads the ImageNet-pretrained backbone, freezes it for Stage 1, and appends a custom 7-class classification head.

In [ ]:
# Load ImageNet Pretrained EfficientNetV2-S Backbone
base_model = tf.keras.applications.EfficientNetV2S(
    include_top=False,
    weights="imagenet",
    input_shape=INPUT_SHAPE
)

# Initially freeze the backbone for Stage 1 feature extraction
base_model.trainable = False

# Build Custom 7-Class Classification Head
inputs = layers.Input(shape=INPUT_SHAPE, name="input_image")
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D(name="avg_pool")(x)
x = layers.BatchNormalization(name="batch_norm")(x)
x = layers.Dropout(0.3, name="top_dropout")(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)

model = models.Model(inputs=inputs, outputs=outputs, name=f"PVision_EfficientNetV2S_{MODALITY.upper()}")

# Compile model for Stage 1
model.compile(
    optimizer=optimizers.Adam(learning_rate=LR_STAGE1),
    loss=losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        metrics.CategoricalAccuracy(name="accuracy"),
        metrics.TopKCategoricalAccuracy(k=2, name="top2_accuracy")
    ]
)

print(f"Model '{model.name}' constructed and compiled for Stage 1.")

## 11. Model Architecture Summary

In [ ]:
model.summary()

## 12. Training Callbacks Configuration
- **ModelCheckpoint:** Saves the best model weights based on minimum validation loss (`val_loss`)
- **EarlyStopping:** Halts training if validation loss stagnates for 5 epochs
- **ReduceLROnPlateau:** Dynamically reduces learning rate when progress plateaus

In [ ]:
best_model_file = WORKING_DIR / "efficientnetv2s_pv_fault_best.keras"

cb_checkpoint = callbacks.ModelCheckpoint(
    filepath=str(best_model_file),
    monitor="val_loss",
    save_best_only=True,
    verbose=1,
    mode="min"
)

cb_early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
    mode="min"
)

cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1,
    mode="min"
)

training_callbacks = [cb_checkpoint, cb_early_stop, cb_reduce_lr]
print(f"Callbacks configured. Checkpoint target: {best_model_file}")

## 13. Stage 1: Feature Extraction Training (Frozen Backbone)
Trains the classification head to adapt to PV fault representations while keeping the core ImageNet feature extraction layers frozen.

In [ ]:
print("=" * 65)
print(">>> COMMENCING STAGE 1: Training Classification Head <<<")
print("=" * 65)

active_weights = class_weights if USE_CLASS_WEIGHTS else None

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    class_weight=active_weights,
    callbacks=training_callbacks,
    verbose=1
)

## 14. Stage 2: Fine-Tuning (Upper Backbone Layers Unfrozen)
Unfreezes the top 50 layers of EfficientNetV2-S and trains with a 10× smaller learning rate ($1 	imes 10^{-4}$).

In [ ]:
print("=" * 65)
print(f">>> COMMENCING STAGE 2: Fine-Tuning Top {UNFREEZE_LAYERS} Layers <<<")
print("=" * 65)

base_model.trainable = True
freeze_cutoff = max(0, len(base_model.layers) - UNFREEZE_LAYERS)

for idx, layer in enumerate(base_model.layers):
    if idx < freeze_cutoff:
        layer.trainable = False
    else:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True

unfrozen_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Unfrozen {unfrozen_count} backbone layers for fine-tuning.")

# Recompile with fine-tuning learning rate
model.compile(
    optimizer=optimizers.Adam(learning_rate=LR_STAGE2),
    loss=losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        metrics.CategoricalAccuracy(name="accuracy"),
        metrics.TopKCategoricalAccuracy(k=2, name="top2_accuracy")
    ]
)

total_epochs = EPOCHS_STAGE1 + EPOCHS_STAGE2

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=len(history_stage1.epoch),
    epochs=total_epochs,
    class_weight=active_weights,
    callbacks=training_callbacks,
    verbose=1
)

## 15. Training Convergence Curves (Accuracy & Loss)
Generates high-resolution convergence plots across both Stage 1 and Stage 2.

In [ ]:
# Merge history metrics across Stage 1 and Stage 2
combined_hist = {}
for k in history_stage1.history.keys():
    s1_vals = list(history_stage1.history[k])
    s2_vals = list(history_stage2.history.get(k, []))
    combined_hist[k] = s1_vals + s2_vals

epochs_axis = range(1, len(combined_hist["loss"]) + 1)

# Plot 1: Accuracy Curve
plt.figure(figsize=(9, 5))
plt.plot(epochs_axis, combined_hist["accuracy"], "b-o", linewidth=2, label="Training Accuracy")
plt.plot(epochs_axis, combined_hist["val_accuracy"], "r--s", linewidth=2, label="Validation Accuracy")
plt.axvline(x=EPOCHS_STAGE1, color="gray", linestyle=":", label="Stage 2 Fine-Tuning Start")
plt.title("EfficientNetV2-S Training and Validation Accuracy", fontsize=12, fontweight="bold")
plt.xlabel("Epoch", fontsize=10, fontweight="bold")
plt.ylabel("Accuracy", fontsize=10, fontweight="bold")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()

plt.savefig(RESULTS_DIR / "training_accuracy.png", dpi=200)
plt.savefig(WORKING_DIR / "training_accuracy.png", dpi=200)
plt.show()

# Plot 2: Loss Curve
plt.figure(figsize=(9, 5))
plt.plot(epochs_axis, combined_hist["loss"], "b-o", linewidth=2, label="Training Loss")
plt.plot(epochs_axis, combined_hist["val_loss"], "r--s", linewidth=2, label="Validation Loss")
plt.axvline(x=EPOCHS_STAGE1, color="gray", linestyle=":", label="Stage 2 Fine-Tuning Start")
plt.title("EfficientNetV2-S Training and Validation Loss", fontsize=12, fontweight="bold")
plt.xlabel("Epoch", fontsize=10, fontweight="bold")
plt.ylabel("Loss", fontsize=10, fontweight="bold")
plt.legend(loc="upper right")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()

plt.savefig(RESULTS_DIR / "training_loss.png", dpi=200)
plt.savefig(WORKING_DIR / "training_loss.png", dpi=200)
plt.show()

## 16. Unseen Test Dataset Evaluation
Evaluates the best checkpoint exclusively on the unseen test partition.

In [ ]:
# Load best model checkpoint
print(f"Loading best checkpoint: {best_model_file}")
best_model = tf.keras.models.load_model(str(best_model_file))

# Run predictions across test set
y_true = test_df["label"].values
y_pred_probs = best_model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# Calculate Core Metrics
test_acc = accuracy_score(y_true, y_pred)
prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")

print("\n" + "=" * 60)
print("            TEST DATASET EVALUATION RESULTS")
print("=" * 60)
print(f"Test Accuracy:        {test_acc * 100:.2f}%")
print(f"Macro Precision:      {prec_macro * 100:.2f}%")
print(f"Macro Recall:         {rec_macro * 100:.2f}%")
print(f"Macro F1-Score:       {f1_macro * 100:.2f}%")
print(f"Weighted F1-Score:    {f1_weighted * 100:.2f}%")
print("=" * 60)

## 17. Detailed Classification Report
Generates precision, recall, and F1-score for each fault category.

In [ ]:
clf_report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
df_clf_report = pd.DataFrame(clf_report_dict).transpose()

# Display formatted table
display(df_clf_report.style.format({
    "precision": "{:.4f}",
    "recall": "{:.4f}",
    "f1-score": "{:.4f}",
    "support": "{:,.0f}"
}))

# Save CSV
df_clf_report.to_csv(RESULTS_DIR / "classification_report.csv")
df_clf_report.to_csv(WORKING_DIR / "classification_report.csv")
print("Saved classification report to /kaggle/working/classification_report.csv")

## 18. Full 7-Class Confusion Matrix Heatmap
Annotated heatmap displaying true versus predicted classifications.

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    linewidths=0.5,
    cbar=True
)
plt.title("EfficientNetV2-S PV Fault Classification — Confusion Matrix", fontsize=12, fontweight="bold")
plt.xlabel("Predicted Class", fontsize=11, fontweight="bold")
plt.ylabel("Actual Class", fontsize=11, fontweight="bold")
plt.xticks(rotation=30, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()

plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=200)
plt.savefig(WORKING_DIR / "confusion_matrix.png", dpi=200)
plt.show()

## 19. Per-Class Performance Metrics
Grouped bar chart displaying Precision, Recall, and F1-Score for each fault category.

In [ ]:
prec_pc, rec_pc, f1_pc, _ = precision_recall_fscore_support(y_true, y_pred, average=None)

df_per_class = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Precision": prec_pc,
    "Recall": rec_pc,
    "F1-Score": f1_pc
})

x_indices = np.arange(NUM_CLASSES)
bar_width = 0.25

plt.figure(figsize=(12, 5.5))
plt.bar(x_indices - bar_width, df_per_class["Precision"], width=bar_width, label="Precision", color="#1f77b4")
plt.bar(x_indices, df_per_class["Recall"], width=bar_width, label="Recall", color="#ff7f0e")
plt.bar(x_indices + bar_width, df_per_class["F1-Score"], width=bar_width, label="F1-Score", color="#2ca02c")

plt.title(f"Per-Class Performance Metrics — {MODALITY.upper()} Modality (EfficientNetV2-S)", fontsize=12, fontweight="bold")
plt.xlabel("PV Fault Classes", fontsize=11, fontweight="bold")
plt.ylabel("Score (0.0 to 1.0)", fontsize=11, fontweight="bold")
plt.xticks(x_indices, CLASS_NAMES, rotation=25, ha="right", fontsize=10)
plt.ylim(0.0, 1.1)
plt.legend(loc="lower right")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()

plt.savefig(RESULTS_DIR / "per_class_metrics.png", dpi=200)
plt.savefig(WORKING_DIR / "per_class_metrics.png", dpi=200)
plt.show()

display(df_per_class)

## 20. Single-Image Prediction & Probability Visualization
Visualizes the original image alongside its predicted 7-class probability distribution.

In [ ]:
def plot_prediction_probabilities(sample_idx: int):
    img_path = test_df.iloc[sample_idx]["file_path"]
    actual_label = test_df.iloc[sample_idx]["class_name"]
    probs = y_pred_probs[sample_idx]
    pred_idx = np.argmax(probs)
    pred_label = CLASS_NAMES[pred_idx]
    confidence = probs[pred_idx] * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    # Left: Image
    img = Image.open(img_path)
    ax1.imshow(img)
    ax1.set_title(f"Actual: {actual_label}\nPredicted: {pred_label} ({confidence:.1f}%)", fontsize=11, fontweight="bold")
    ax1.axis("off")

    # Right: Horizontal Probability Bar Chart
    colors_list = ["#2ca02c" if c == pred_label else "#1f77b4" for c in CLASS_NAMES]
    y_pos = np.arange(NUM_CLASSES)
    ax2.barh(y_pos, probs * 100, color=colors_list, edgecolor="black", alpha=0.85)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(CLASS_NAMES, fontsize=10)
    ax2.set_xlabel("Probability (%)", fontsize=10, fontweight="bold")
    ax2.set_title("Predicted Class Probabilities", fontsize=11, fontweight="bold")
    ax2.set_xlim(0, 100)
    ax2.grid(axis="x", linestyle="--", alpha=0.6)

    for i, p in enumerate(probs):
        ax2.text(p * 100 + 1, i, f"{p * 100:.1f}%", va="center", fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.show()

# Visualize 2 sample test predictions
for i in [0, min(10, len(test_df) - 1)]:
    plot_prediction_probabilities(i)

## 21. Sample Prediction Grid
Displays test predictions with visual indication for correct (green) and incorrect (red) classifications.

In [ ]:
num_grid = min(12, len(test_df))
fig, axes = plt.subplots(3, 4, figsize=(16, 11))
axes = axes.flatten()

for i in range(num_grid):
    img_path = test_df.iloc[i]["file_path"]
    actual_name = test_df.iloc[i]["class_name"]
    pred_name = CLASS_NAMES[y_pred[i]]
    conf = y_pred_probs[i][y_pred[i]] * 100
    is_correct = (y_true[i] == y_pred[i])

    img = Image.open(img_path)
    axes[i].imshow(img)
    
    title_color = "green" if is_correct else "red"
    symbol = "✓" if is_correct else "✗"
    axes[i].set_title(
        f"{symbol} Actual: {actual_name}\nPred: {pred_name} ({conf:.1f}%)",
        fontsize=9,
        fontweight="bold",
        color=title_color
    )
    axes[i].axis("off")

plt.suptitle("EfficientNetV2-S Test Predictions Grid (Green: Correct | Red: Misclassified)", fontsize=13, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

## 22. Save Model, Class Names, and Training History
Exports all trained model artifacts and JSON metadata to `/kaggle/working/` for download.

In [ ]:
# Save Class Names JSON
class_names_path = WORKING_DIR / "class_names.json"
with open(class_names_path, "w", encoding="utf-8") as f:
    json.dump(CLASS_NAMES, f, indent=4)
print(f"Saved: {class_names_path}")

# Save Training History JSON
history_path = WORKING_DIR / "training_history.json"
with open(history_path, "w", encoding="utf-8") as f:
    json.dump(combined_hist, f, indent=4)
print(f"Saved: {history_path}")

print(f"Best Trained Model: {best_model_file}")

## 23. Final Results Summary Table
Executive summary table calculated directly from the test evaluation metrics (no fabricated values).

In [ ]:
final_summary_rows = [
    {"Metric": "Test Accuracy", "Score": f"{test_acc * 100:.2f}%"},
    {"Metric": "Macro Precision", "Score": f"{prec_macro * 100:.2f}%"},
    {"Metric": "Macro Recall", "Score": f"{rec_macro * 100:.2f}%"},
    {"Metric": "Macro F1-Score", "Score": f"{f1_macro * 100:.2f}%"},
    {"Metric": "Weighted F1-Score", "Score": f"{prec_weighted * 100:.2f}%"},
    {"Metric": "Total Test Samples", "Score": f"{len(test_df):,}"}
]

df_final_summary = pd.DataFrame(final_summary_rows)
print("=" * 45)
print("           FINAL RESULTS SUMMARY TABLE")
print("=" * 45)
display(df_final_summary)
print("=" * 45)

## 24. Kaggle Output Files Verification
Lists all generated artifacts ready for download from the `/kaggle/working/` directory.

In [ ]:
print("Files generated in /kaggle/working/ ready for download:")
print("-" * 55)
for item in sorted(WORKING_DIR.iterdir()):
    if item.is_file():
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f"  ✓ {item.name:<35} ({size_mb:>6.2f} MB)")
print("-" * 55)